# Advanced IR: Cascade Rankers (Part 1)

In the first part of this tutorial, you will learn about cascade ranking models. You will combine two ranking models to (hopefully!) improve retrieval effectiveness.

### Preparation: Install dependencies

First, we install the libraries

In [ ]:
!pip install python-terrier lightning_ir pyterrier-alpha tqdm pandas

Now, let's start with msmarco, instead of our scifacts corpus.

In [ ]:
dataset_name = "nano-beir/msmarco"

In [ ]:
import os
from pathlib import Path

import torch

from lightning_ir import TorchDenseIndexConfig, TorchDenseSearchConfig
from lightning_ir_pyterrier import BiEncoder, CrossEncoder

from pyterrier.datasets import get_dataset
from pyterrier.text import get_text
import pyterrier as pt

import pandas as pd
pd.set_option('max_colwidth', 400)

from pyterrier.datasets import get_dataset
dataset = get_dataset(f"irds:{dataset_name}")

device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
if torch.backends.mps.is_available():
    device = "mps"

Last week, we called this a neural ranker. In reality this is a bi-encoder model. We will learn more about this model architecture next week.

In [ ]:
index_dir = Path(f"./vector_index_{dataset_name.replace('/','-')}")

bi_encoder = BiEncoder(
    model_name_or_path="webis/bert-bi-encoder",
    index_dir=index_dir,
    index_config=TorchDenseIndexConfig(),
    search_config=TorchDenseSearchConfig(k=100),
    batch_size=32,
    verbose=True,
    use_gpu=True
)
bi_encoder._module.to(device)
bi_encoder._module.device

This week, you will also get to play around with a new type of neural model: a cross-encoder.

In [ ]:
cross_encoder = CrossEncoder(
    model_name_or_path="webis/monoelectra-base",
    batch_size=1, # Due to a bug, this can only be a value less than and a multiple of the number of documents to re-rank. The slowest but most accurate result is simply scoring each document one by one, i.e., batch_size=1
    verbose=True,
    use_gpu=True
)
if torch.cuda.is_available():
    bi_encoder._module.to(device)
    bi_encoder._module.device

Both of these models use a transformer architecture to represent queries and documents. In specific, both use just a single encoder model.

### Step 1: Indexing

First we need to index the data using out bi-encoder. Our this type of model pre-computes representations for documents offline and only needs to compute representations for queries at inference time. This pre-processing step makes bi-encoders much more efficient than cross-encoders at the cost of lower ranking effectivness.

In [ ]:
from pathlib import Path
if not index_dir.exists():
    bi_encoder.index(dataset.get_corpus_iter())

Since we want to compare the model to the models from last week, we need to index the data in pyterrier too.

In [ ]:
pt_index_path = f"./index_{dataset_name.replace('/','-')}"

if not os.path.exists(pt_index_path + "/data.properties"):
  # create the index, using the IterDictIndexer indexer 
  indexer = pt.index.IterDictIndexer(pt_index_path, meta_reverse=[])

  # we give the dataset get_corpus_iter() directly to the indexer
  # while specifying the fields to index and the metadata to record
  index_ref = indexer.index(dataset.get_corpus_iter())

else:
  # if you already have the index, use it.
  index_ref = pt_index_path

index = pt.IndexFactory.of(index_ref)

tfidf = pt.BatchRetrieve(index, wmodel="TF_IDF")
bm25 = pt.BatchRetrieve(index, wmodel="BM25")

### Step 2: Search

Now, we can see some results, just like last week. Let's look at just the first three topics, and the top 100 documents scored by the bi-encoder. Notice how fast this retrieval and scoring process is.

In [ ]:
from pandas import DataFrame

topics = dataset.get_topics(variant="text").iloc[:3]
bi_encoder.transform(topics)

Cross-encoder models are actually so much more computationally intensive that in almost all cases, they are only ever used to re-rank a portion of an existing ranking. Gradually re-ranking an initial ranking using progressively more expensive ranking models is called a ranking cascade or a cascade model (think something similar to an ensemble model in other machine learning scenarios).

In pyterrier, we can easily get the top-k documents from a ranking using the `%k` syntax. 

**NOTE** the pyterrier `Transformer` class just happens to be an (unfortunate) name clash (naming things is difficult!), so the transformer below is just a part of pyterrier that "transforms" dataframes using retrieval and ranking functions.

In [ ]:
pipe1 = pt.Transformer.from_df(bi_encoder.transform(topics)) %10 
pipe1.transform(topics)

The cross-encoder must encode every query and document pair at inference time, making it considerably more slower than the bi-encoder (and the model being used here is just a "small" one!).

That means we need to grab the text content from each document we need to re-rank. Fortunately, pyterrier provides us with a handy syntax to do that: the composition operator (`>>`). This operator sends the output of one pipeline (the left hand side) into the input pipeline (the right hand side).

In [ ]:
pipe2 = pipe1 >> get_text(dataset)
pipe2.transform(topics)

Now that we have the text content of the documents we want to score using the cross-encoder, we can simply add compose it with our pipeline.

Take care to note how much longer it takes to re-rank the documents from the bi-encoder than retrieving 100 documents using the bi-encoder. Take some time to increase the top-k you're re-ranking above to see this effect get larger if it seems to slow for you.

In [ ]:
pipe3 = pipe2 >> cross_encoder
pipe3.transform(topics).sort_values(by=["index", "rank"])

Take a moment to compare the ranking from the bi-encoder to the cross-encoder. What do you think? Do you think the results improved? Where did you notice differences?

### Step 3: Evaluation



We can now compare the three retrieval models against each other.

In [ ]:
from ir_measures import nDCG, RR, MAP
from pyterrier.pipelines import Experiment
from pyterrier.text import get_text

Experiment(
    [
        tfidf,
        bm25,
        bi_encoder,
        bi_encoder % 10 >> get_text(dataset) >> cross_encoder,
    ],
    dataset.get_topics(variant='text'),
    dataset.get_qrels(),
    eval_metrics=[nDCG@10, MAP, RR],
    names=[
        "tf-idf",
        "BM25",
        "bi-encoder",
        "bi-encoder >> cross-encoder",
    ],
    verbose=True,
)

### Step 4: Other Datasets

You should see that the results of the cascade model (the cross-encoder re-ranking the bi-encoder) do indeed improve retrieval effectiveness. But was this a fluke? Try doing these experiments again on the scifacts dataset, or start playing around with datasets from https://ir-datasets.com/. Note that on very large datasets you may wish to use bm25 as your first stage retrieval model instead of the bi-encoder since encoding large datasets (>1M documents/passages) will take a very long time on the hardware you have available.

When you are ready, move onto Part 2 of the tutorial.